# Step 4 Topic Modeling Results Analysis

In [12]:
import pandas as pd
from bertopic import BERTopic

## 1 Read in Data, Model and Topics
### 1.1 Load Topic Model

In [7]:
topic_model = BERTopic.load("results/model_tfidf_model")

In [9]:
#get the topics frequencies
freq = topic_model.get_topic_info()
freq

,Topic,Count,Name,Representation,Representative_Docs
0,-1,923,-1_development_planning_building_process,"[development, planning, building, process, pol...",[decision maker facing unprecedented challenge...
1,0,453,0_gis_land_ca_growth,"[gis, land, ca, growth, automaton, cellular, p...",[zone cell parcel long regarded main unit anal...
2,1,151,1_accessibility_travel_cycling_commuting,"[accessibility, travel, cycling, commuting, bi...",[people value enjoy benefit use car brings qua...
3,2,81,2_design_style_designer_architectural,"[design, style, designer, architectural, archi...",[two experiment reported examine hypothesis co...
4,3,76,3_network_centrality_street_topology,"[network, centrality, street, topology, traffi...",[transportation research ha usually seen road ...
5,4,52,4_homeless_social_tourist_mobile,"[homeless, social, tourist, mobile, mobility, ...",[mobile internet emerges numerous instagram wo...
6,5,49,5_rectangular_adjacency_room_dissection,"[rectangular, adjacency, room, dissection, max...",[procedure described generates dissection rect...
7,6,49,6_facility_heuristic_problem_solution,"[facility, heuristic, problem, solution, cover...",[previous research effort demonstrate use loca...
8,7,48,7_energy_heat_temperature_canopy,"[energy, heat, temperature, canopy, carbon, tr...",[local climate change due urbanization epitomi...
9,8,47,8_segregation_ethnic_tract_residential,"[segregation, ethnic, tract, residential, gend...",[segregation highly nuanced concept researcher...


In [10]:
# Get all word representing each topic from topic model
from src.analysistools import __get_topic_allwords__
topic_allwords = __get_topic_allwords__(freq.Topic, topic_model)
topic_allwords

,Topic,Freq,Words
0,Topic-1,923,"[development, planning, building, process, pol..."
1,Topic0,453,"[gis, land, ca, growth, automaton, cellular, p..."
2,Topic1,151,"[accessibility, travel, cycling, commuting, bi..."
3,Topic2,81,"[design, style, designer, architectural, archi..."
4,Topic3,76,"[network, centrality, street, topology, traffi..."
5,Topic4,52,"[homeless, social, tourist, mobile, mobility, ..."
6,Topic5,49,"[rectangular, adjacency, room, dissection, max..."
7,Topic6,49,"[facility, heuristic, problem, solution, cover..."
8,Topic7,48,"[energy, heat, temperature, canopy, carbon, tr..."
9,Topic8,47,"[segregation, ethnic, tract, residential, gend..."


### 1.2 Load Topic Overtime dataframe

In [11]:
topics_over_time = pd.read_csv("results/topics_over_time_tfidf_model.csv").iloc[:,1:]
topics_over_time.head()

,Topic,Words,Frequency,Timestamp,Name
0,-1,"algebraic, simplicial, exterior, averaging, ma...",6,1974,-1_development_planning_building_process
1,2,"wright, froebel, sullivan, kind, office",2,1974,2_design_style_designer_architectural
2,6,"teaching, timetabling, minimum, demand, utilis...",1,1974,6_facility_heuristic_problem_solution
3,14,"regulation, bonus, sydney, floor, return",1,1974,14_architect_regulation_control_deregula...
4,18,"heating, appliance, heat, costing, conservation",1,1974,18_nondomestic_stock_energy_building


### 1.3 Load Abstract Data

In [ ]:
###################
# Part 2 Stack Bars
###################
import matplotlib.pyplot as plt

# * Step1: Get the dataframe for the visualization
def GetTopicCountByYear(topiclist, evo_table, yearlycount):
    # extract the key info from eveolution table
    temp_df = evo_table.loc[:, ['Topic', 'Frequency', 'Timestamp']]
    # print(temp_df)

    # Loop through the keywords list to draw lins on Cavas
    topiclist.sort()
    # print("TopicList", topiclist)
    legend_list = []

    # Create a dataframe to hold the all topics, freq, year amount
    stack_df = pd.DataFrame()

    for item in range(0, len(topiclist)):
        topic = topiclist[item]
        topic_label = "Topic " + str(topic)
        # print("Topic", topic)
        legend_list.append("Topic" + str(topic))

        # Extract visulize dataframe
        visual_df = temp_df[temp_df.Topic == topic].reset_index(drop=True)
        visual_df = visual_df.rename(columns={'Timestamp': 'Publication Year'})  # rename timstamp to year fro merge
        #print(visual_df.head())
        #print(yearcount.head())
        # Merge on year
        final_visual_df = pd.merge(visual_df, yearlycount, on="Publication Year")
        # print(final_visual_df.head())
        #stack_df = stack_df.append(final_visual_df)
        stack_df = pd.concat([stack_df, final_visual_df])

    # group the df based on year and topics
    #grouped_stack = stack_df.groupby(["year", "Topic"])["Frequency"].sum().reset_index()
    grouped_stack = stack_df.groupby(["Publication Year", "Topic"])["Frequency"].sum().reset_index()
    # pivot the dataframe into the correct format
    grouped_stack_df = grouped_stack.pivot(index='Publication Year', columns='Topic', values='Frequency').fillna(0).reset_index()
    # print(grouped_stack_df.head())

    return grouped_stack_df


# * Step2: Satcked bar visualization
# Get the topic evo for each topic
def GetTopicEvo(topiclist, overtime):  # topics list, topic_model, topic over time dataframe
    finaldataframe = pd.DataFrame()
    for topic in topiclist:
        # print(topic)
        tempdf = overtime[overtime.Topic == topic].reset_index(drop=True)
        # print(tempdf.head())
        # finaldataframe = finaldataframe.append(tempdf)
        finaldataframe = pd.concat([finaldataframe, tempdf])
    return finaldataframe


def VisualizeStackByYearColor(data, year_range, PlotTitle):  # 3String
    plt.style.use('default')

    # create the figure
    fig, ax = plt.subplots(figsize=(25, 10))
    # plt.figure(figsize=(25,10))
    plt.xlabel('Year', fontsize=30, fontname="Arial")
    plt.ylabel('Frequency', fontsize=30, fontname="Arial")

    print(len(data.columns[1:]))
    if len(data.columns[1:]) == 2:
        color_list = ['#4E62AB', '#FDB96A']

    if len(data.columns[1:]) == 3:
        color_list = ['#4E62AB', '#FDB96A', '#D6404E']

    if len(data.columns[1:]) == 4:
        color_list = ['#4E62AB', '#87CFA4', '#FDB96A', '#D6404E']

    if len(data.columns[1:]) == 5:
        color_list = ['#4E62AB', '#87CFA4', '#F5FBB1', '#FDB96A', '#D6404E']

    if len(data.columns[1:]) == 6:
        color_list = ['#4E62AB', '#87CFA4', '#CBE99D', '#FEE89A', '#F57547', '#9E0142']

    if len(data.columns[1:]) == 7:
        color_list = ['#4E62AB', '#469EB4', '#87CFA4', '#FEE89A', '#FDB96A', '#F57547', '#9E0142']

    if len(data.columns[1:]) == 8:
        color_list = ['#4E62AB', '#469EB4', '#87CFA4', '#F5FBB1', '#FEE89A', '#FDB96A', '#F57547', '#9E0142']

    if len(data.columns[1:]) == 9:
        color_list = ['#4E62AB', '#469EB4', '#87CFA4', '#F5FBB1', '#FEE89A', '#FDB96A', '#F57547', '#D6404E', '#9E0142']

    # Legend of the stacked bars
    bottom = 0
    colorid = 0
    legend_list = []
    for topic in data.columns[1:]:
        # legend
        topic_label = "Topic " + str(topic)
        # print("Topic", topic)
        legend_list.append("Topic" + str(topic))
        # bar plot
        # plt.bar(data.loc[:, 'year'], data[topic], bottom = bottom, width=0.95)
        plt.bar(data.index, data[topic], bottom=bottom, width=0.95, color=color_list[colorid])
        bottom = bottom + data[topic]
        colorid = colorid + 1

    # ax.set_title(PlotTitlem)
    plt.legend((legend_list), prop={'family': 'Arial', "size": 20}, loc='upper left', ncol=1)

    # set x ticks: dont' show the year without any topics
    #x_label = list(np.arange(data.year.min(), 2021, 5))
    #x_label.append(2021)
    #x_label = list(data.year)
    x_label = list(data["Publication Year"])
    print(x_label)

    x_ticks = []
    for year_index in range(0, len(x_label)):
        #print(year)
        year = x_label[year_index]
        try:
            temp = data[data['Publication Year'] == year].index[0]
            #print(temp)
        except:
            temp = data[data['Publication Year'] == year + 1].index[0]
        #if temp%5 == 0:
        x_ticks.append(temp)
    print(x_ticks)

    # x_ticks = data.index.to_list()
    # x_label = data.loc[:, 'year'].to_list()
    # x_label = np.arange(1975, 2021, 5)
    # Plot x and y axticks
    plt.gcf().autofmt_xdate()  # italics of x label
    plt.xticks(ticks=x_ticks, labels=x_label, fontsize=15)
    # plt.xticks(np.arange(1975, 2021, 5))
    plt.yticks(fontsize=25)
    plt.margins(x=0.01)
    plt.show()

def _visualsmall_(data, year_range):
    plt.style.use('default')

    print(len(data.columns[1:]))
    if len(data.columns[1:]) == 2:
        color_list = ['#4E62AB', '#FDB96A']

    if len(data.columns[1:]) == 3:
        color_list = ['#4E62AB', '#FDB96A', '#D6404E']

    if len(data.columns[1:]) == 4:
        color_list = ['#4E62AB', '#87CFA4', '#FDB96A', '#D6404E']

    if len(data.columns[1:]) == 5:
        color_list = ['#4E62AB', '#87CFA4', '#F5FBB1', '#FDB96A', '#D6404E']

    if len(data.columns[1:]) == 6:
        color_list = ['#4E62AB', '#87CFA4', '#CBE99D', '#FEE89A', '#F57547', '#9E0142']

    if len(data.columns[1:]) == 7:
        color_list = ['#4E62AB', '#469EB4', '#87CFA4', '#FEE89A', '#FDB96A', '#F57547', '#9E0142']

    if len(data.columns[1:]) == 8:
        color_list = ['#4E62AB', '#469EB4', '#87CFA4', '#F5FBB1', '#FEE89A', '#FDB96A', '#F57547', '#9E0142']

    if len(data.columns[1:]) == 9:
        color_list = ['#4E62AB', '#469EB4', '#87CFA4', '#F5FBB1', '#FEE89A', '#FDB96A', '#F57547', '#D6404E', '#9E0142']
    # ax = plt.axes()
    if year_range > 0:
        # figure and axis
        fig, ax = plt.subplots(1, figsize=(16, 9))
        plt.xlabel('Year', fontsize=20, fontname="Arial")

        print("Small plot is the topic before the year of ", year_range)
        #year_index = data[data['year'] == year_range].index[0]
        year_index = data[data['Publication Year'] == year_range].index[0]
        # print(year_index.tolist())
        small_df = data.iloc[:year_index, :]  # .set_index('year')
        # print(small_df.tail())

        bottom = 0
        colorid = 0
        for topic in data.columns[1:]:
            # bar plot
            # plt.bar(small_df.loc[:, 'year'], small_df[topic], bottom = bottom, width=0.95)
            plt.bar(small_df.index, small_df[topic], bottom=bottom, width=0.95, color=color_list[colorid])
            bottom = bottom + small_df[topic]
            colorid = colorid + 1

        # remove spines
        # ax.spines['right'].set_visible(False)
        # ax.spines['left'].set_visible(False)
        # ax.spines['top'].set_visible(False)
        # ax.spines['bottom'].set_visible(False)
        x_ticks = small_df.index
        #x_label = small_df.loc[:, 'year'].to_list()
        x_label = small_df.loc[:, 'Publication Year'].to_list()

        plt.gcf().autofmt_xdate()  # italics of x label
        plt.xticks(ticks=x_ticks, labels=x_label, fontsize=15)
        plt.yticks(fontsize=25)
        plt.margins(x=0.01)
        plt.show()


###############################END OF PART TWO#############################################
